#### **Importation des bibliothèques**

In [1]:
# Utilitaires de base
import builtins
import pandas as pd

# Suivi des expriences (MLflow & DagsHub)
import dagshub
import mlflow

# Scikit-Learn : Sparation des donnes et mtriques d'valuation
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    f1_score, 
    fbeta_score, 
    precision_score, 
    recall_score
)

# TensorFlow / Keras : Cration et entranement du modle Deep Learning
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import TextVectorization

#### **DagsHub & MLflow Init**

In [ ]:

# Initialisation de la connexion DagsHub avec les identifiants de votre dépôt et activation du mode MLflow
# PATCH WINDOWS : Force l'utilisation de l'encodage UTF-8 lors de l'écriture des fichiers.
# Ceci corrige l'erreur "charmap codec can't encode characters" causée par le nouveau format d'affichage (summary) de Keras 3
import builtins
_original_open = builtins.open
def _utf8_open(*args, **kwargs):
    mode = kwargs.get('mode', args[1] if len(args) > 1 else 'r')
    if 'b' not in mode and 'encoding' not in kwargs:
        kwargs['encoding'] = 'utf-8'
    return _original_open(*args, **kwargs)
builtins.open = _utf8_open

dagshub.init(repo_owner='Oscar-AS', repo_name='disaster-tweets-project', mlflow=True)

# Définition du nom du dossier (expérience) dans MLflow où toutes nos métriques seront classées
mlflow.set_experiment("Disaster_Tweets")

# Affichage d'un message console pour confirmer que le tracking est bien connecté
print("MLflow activé avec succès sur DagsHub !")


Accessing as Oscar-AS

Initialized MLflow to track repo "Oscar-AS/disaster-tweets-project"

Repository Oscar-AS/disaster-tweets-project initialized!

MLflow activé avec succès sur DagsHub !


#### Importation des données


In [4]:
# Chargement des données
# Lecture du fichier CSV depuis le dossier Base et stockage dans la variable 'df'
df = pd.read_csv("Base/tweets.csv")



#### **Séparation des données**

In [5]:
# Séparation Train/Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['target'], test_size=0.2, random_state=42, stratify=df['target']
)

# Affiche dans la console le nombre de tweets utilisés pour l'entraînement
print(f"Taille de l'entraînement : {len(X_train)}")
# Affiche dans la console le nombre de tweets gardés pour le test
print(f"Taille du test : {len(X_test)}")

Taille de l'entraînement : 9096
Taille du test : 2274


#### **Modèles Transformers via Hugging Face**

In [6]:
# Import de l'objet Dataset de Hugging Face
from datasets import Dataset
# Import de evaluate pour calculer de façon standardisée les métriques d'évaluation
# Import de NumPy
import numpy as np

# Transformation de notre DataFrame d'entraînement Pandas en un objet "Dataset" ultra-optimisé de Hugging Face
hf_train = Dataset.from_pandas(pd.DataFrame({'text': X_train, 'label': y_train}))
# Transformation de notre DataFrame de test Pandas
hf_test = Dataset.from_pandas(pd.DataFrame({'text': X_test, 'label': y_test}))

# Importation de scikit-learn pour calculer facilement le F2-Score et les métriques par classe
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, fbeta_score

# Fonction exécutée à la fin de chaque Epoch par le Trainer pour calculer le score
def compute_metrics(eval_pred):
    # Séparation des probabilités prédites (logits) et des vraies réponses (labels)
    logits, labels = eval_pred
    # L'argmax récupère la classe ayant reçu la plus forte probabilité (0 ou 1)
    predictions = np.argmax(logits, axis=-1)
    
    # Précision et Rappel par classe
    precision_cls = precision_score(labels, predictions, average=None)
    recall_cls = recall_score(labels, predictions, average=None)
    
    # Calcul des métriques globales
    f1 = f1_score(labels, predictions, average="macro")
    f2 = fbeta_score(labels, predictions, beta=2, average="macro")
    accuracy = accuracy_score(labels, predictions)
    
    # Retourner toutes les métriques pour le suivi MLflow (le Trainer ajoutera automatiquement le préfixe "eval_")
    return {
        "f1_macro": f1,
        "f2_score": f2,
        "precision_class_0": precision_cls[0],
        "precision_class_1": precision_cls[1],
        "recall_class_0": recall_cls[0],
        "recall_class_1": recall_cls[1],
        "accuracy": accuracy
    }

# Importation du système d'exploitation
import os
# Paramétrage de la variable d'environnement qui indique à Hugging Face dans quel dossier MLflow il doit écrire
os.environ["MLFLOW_EXPERIMENT_NAME"] = "Disaster_Tweets_Niveau_3_et_4"
# Augmente le délai à 10 minutes pour éviter les erreurs d'upload sur DagsHub
os.environ["MLFLOW_HTTP_REQUEST_TIMEOUT"] = "600"


In [7]:
# Import des AutoClasses (la magie de Hugging Face pour importer n'importe quel modèle du web en 1 ligne)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

# Définition d'une fonction Python réutilisable pour entraîner n'importe quel Transformer sans réécrire le code
def train_hf_model(model_id, run_name, batch_size=16, epochs=2):
    # Affichage du démarrage
    print(f"========== Début de l'entraînement pour {model_id} ==========")
    
    # 1. Chargement du Tokenizer spécifique au modèle (le dictionnaire qui transforme les mots en IDs)
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    # Fonction qui applique le tokenizer sur une phrase
    def tokenize_function(examples):
        # On coupe les phrases (truncation=True) à 128 "tokens" maximum et on ajoute du vide (padding) pour les plus courtes
        return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)
    
    # Application massive et extrêmement rapide (batched=True) du Tokenizer sur tout le jeu d'entraînement
    tokenized_train = hf_train.map(tokenize_function, batched=True)
    # Même chose pour le test
    tokenized_test = hf_test.map(tokenize_function, batched=True)
    
    # 2. Chargement de l'architecture du Transformer avec une tête de classification pour 2 sorties (0 ou 1)
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
    
    # 3. Paramètres de l'entraînement 
    training_args = TrainingArguments(
        output_dir=f"./results_{run_name}",  # Dossier de sauvegarde
        eval_strategy="epoch",         # Evaluer le modèle à la fin de chaque Epoch
        save_strategy="epoch",               # Sauvegarder un "point de contrôle" à la fin de chaque Epoch
        learning_rate=2e-5,                  # Taux d'apprentissage très petit (spécifique aux Transformers)
        per_device_train_batch_size=batch_size, # Taille des paquets envoyés à la carte graphique (entraînement)
        per_device_eval_batch_size=batch_size,  # Taille des paquets envoyés à la carte graphique (test)
        num_train_epochs=epochs,             # Nombre total d'itérations
        weight_decay=0.01,                   # Ajout de pénalités pour éviter le surapprentissage
        load_best_model_at_end=True,         # A la fin, on recharge la version qui a eu le meilleur score
        report_to="mlflow",                  # Dit au système d'envoyer tout le suivi de cet entraînement vers MLflow (DagsHub)
        run_name=run_name,                   # Nom du run dans l'interface MLflow
    )
    
    # 4. L'Objet Trainer qui s'occupe de gérer toute la boucle mathématique PyTorch en arrière-plan
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_test,
        compute_metrics=compute_metrics, # On utilise notre fonction personnalisée pour mesurer le F1-Score
    )
    
    # Démarre l'entraînement intensif
    trainer.train()

    # --- NOUVEAU : Sauvegarde locale de sécurité ---
    save_path = f"./best_model_{run_name}"
    print(f"Sauvegarde du modèle en local dans {save_path}...")
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)

    
    try:
        print("Tentative d'envoi du modèle vers DagsHub...")
        components = {"model": trainer.model, "tokenizer": tokenizer}
        mlflow.transformers.log_model(
            transformers_model=components, 
            artifact_path="model",
            task="text-classification"
        )
        print("✅ Modèle enregistré avec succès sur DagsHub !")
        # Tente d'enregistrer le modèle HuggingFace dans MLflow pour la mise en production
    except Exception as e:
        print(f"⚠️ L'envoi vers DagsHub a échoué (souvent dû à la taille du fichier) : {e}")
        print(f"Pas de souci, ton modèle est bien sauvegardé ici : {save_path}")
        print("Avertissement: L'enregistrement du modèle Transformers dans MLflow a échoué:", e)
    
    # Force MLflow à fermer proprement la session de suivi de ce run
    mlflow.end_run()
    # Affiche la fin dans la console
    print(f"========== Fin de l'entraînement pour {model_id} ==========\n")
    return trainer, tokenized_test



#### **Modèle BERT (Bidirectional Encoder Representations from Transformers)**

##### **Description du modèle**
Le modèle révolutionnaire publié par Google en 2018 qui a changé l'histoire du NLP.

##### **Explication du fonctionnement**
Contrairement aux anciens modèles qui lisaient le texte de gauche à droite, BERT utilise le **Mécanisme d'Attention**. Il regarde *tous* les mots de la phrase simultanément pour comprendre la relation exacte de chaque mot par rapport à tous les autres mots. (Le fameux "Contexte Bidirectionnel Profond"). Il a été pré-entraîné en lisant tout Wikipédia.


In [ ]:
# Appel de la fonction pour entraîner le modèle BERTtweet, version vinai
trainer, tokenized_test = train_hf_model(model_id="vinai/bertweet-base", run_name="BERTweet", epochs=2)



========== Début de l'entraînement pour vinai/bertweet-base ==========


config.json:   0%|          | 0.00/558 [00:00<?, ?B/s]

c:\Users\Dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\models--vinai--bertweet-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/543M [00:00<?, ?B/s]

c:\Users\Dell\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, auc
import tensorflow as tf

# 1. Obtenir les prédictions et probabilités
preds = trainer.predict(tokenized_test)
logits = preds.predictions
y_probs = tf.nn.softmax(logits, axis=-1).numpy()[:, 1]
y_true = preds.label_ids

# 2. Courbe Précision-Rappel
precision, recall, _ = precision_recall_curve(y_true, y_probs)
pr_auc = auc(recall, precision)

plt.figure(figsize=(8, 6))
plt.plot(recall, precision, label=f'PR Curve (AUC = {pr_auc:.2f})', color='b', lw=2)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Courbe Précision-Rappel (BERT)')
plt.legend(loc='lower left')
plt.grid(True)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 3. Courbe de Lift
data = pd.DataFrame({'y_true': y_true, 'y_prob': y_probs})
data = data.sort_values(by='y_prob', ascending=False)
data['cumulative_data_fraction'] = np.arange(1, len(data) + 1) / len(data)
data['cumulative_positive_rate'] = data['y_true'].cumsum() / data['y_true'].sum()
data['lift'] = data['cumulative_positive_rate'] / data['cumulative_data_fraction']

plt.figure(figsize=(8, 6))
plt.plot(data['cumulative_data_fraction'], data['lift'], label='Lift Curve', color='orange', lw=2)
plt.axhline(y=1, color='r', linestyle='--', label='Baseline (Random)')
plt.xlabel('Fraction of data')
plt.ylabel('Lift')
plt.title('Courbe de Lift (BERT)')
plt.legend(loc='upper right')
plt.grid(True)
plt.show()